# Exercise 3.1 - Efficient Fine-tuning for Sentiment Analysis

Kernel usato: `DLA2026-transformers`.

Obiettivo: ridurre il costo del fine-tuning rispetto al notebook 2, dove abbiamo aggiornato tutto DistilBERT.

Confrontiamo due strategie efficienti senza cambiare la pipeline generale:

1. **LoRA**, tramite Hugging Face PEFT, applicata alle proiezioni di attenzione `q_lin` e `v_lin`;
2. **partial freezing**, congelando embeddings e i primi 4 layer transformer.

Il confronto non guarda solo accuracy/F1, ma anche la percentuale di parametri addestrabili.

> **Execution note**
>
> Gli output visibili sono stati prodotti durante le esecuzioni finali o di validazione del laboratorio. Nella versione di consegna i training costosi sono disattivati di default quando sono controllati da flag; checkpoint e artefatti salvati vengono usati per consultazione rapida.

In [1]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

PROJECT_ROOT

WindowsPath('C:/Users/checc/OneDrive/Desktop/DLA/DLA_Lab/DLA_Lab2')

In [2]:
import importlib
import math

import pandas as pd
import torch

import dla_lab2.sentiment as sentiment
from dla_lab2.paths import output_dir
from dla_lab2.seed import set_seed

# Ricarichiamo gli helper per usare le correzioni su Trainer, warmup e PEFT.
importlib.reload(sentiment)

from dla_lab2.sentiment import (
    build_trainer,
    build_training_arguments,
    count_parameters,
    disable_notebook_progress_callback,
    load_distilbert_base,
    load_rotten_tomatoes,
    lora_sequence_classifier_init,
    partial_freezing_sequence_classifier_init,
    suppress_subprocess_reader_unicode_errors,
    tokenize_dataset_splits,
)

set_seed(42)
suppress_subprocess_reader_unicode_errors()

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [3]:
# Stessa preparazione dati del notebook 2.
# Tokenizziamo una volta sola e lasciamo il padding dinamico al data collator del Trainer.
ds_dict = load_rotten_tomatoes()
tokenizer, _ = load_distilbert_base(device=None)
tokenized = tokenize_dataset_splits(ds_dict, tokenizer, max_length=256)

num_epochs = 3
batch_size = 32
total_steps = math.ceil(len(tokenized["train"]) / batch_size) * num_epochs
warmup_steps = math.ceil(total_steps * 0.1)

print(f"Total training steps: {total_steps}")
print(f"Warmup steps: {warmup_steps}")

c:\Users\checc\anaconda3\envs\DLA2026-transformers\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 6575.38it/s]
DistilBertModel LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Map: 100%|██████████| 1066/1066 [00:00<00:00, 11015.15 examples/s]


Total training steps: 801
Warmup steps: 81


## Strategia A - LoRA

LoRA aggiunge piccole matrici addestrabili a basso rango, invece di aggiornare tutti i pesi del transformer. Qui applichiamo LoRA a `q_lin` e `v_lin`, cioe' due proiezioni dell'attenzione di DistilBERT.

Best practice usate:

- backbone pre-addestrato lasciato quasi completamente congelato;
- learning rate piu' alto (`1e-3`) perche' si addestrano pochi parametri nuovi;
- stesso dataset, tokenizer, data collator, metriche e Trainer del notebook 2;
- warmup espresso con `warmup_steps` per evitare warning di deprecazione.

In [4]:
def lora_model_init():
    return lora_sequence_classifier_init(rank=8, alpha=16, dropout=0.1)


lora_args = build_training_arguments(
    output_dir=output_dir("distilbert_lora"),
    learning_rate=1e-3,
    train_batch_size=batch_size,
    eval_batch_size=batch_size,
    num_train_epochs=num_epochs,
    weight_decay=0.01,
    warmup_steps=warmup_steps,
)

lora_trainer = build_trainer(
    tokenizer=tokenizer,
    training_args=lora_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    model_init=lora_model_init,
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 6094.42it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [5]:
lora_train_output = lora_trainer.train()
disable_notebook_progress_callback(lora_trainer)
lora_test = lora_trainer.evaluate(tokenized["test"])
lora_params = count_parameters(lora_trainer.model)
lora_test, lora_params

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 4516.95it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`use_return_dict` is deprecated! Use `return_dict` instead!


({'eval_loss': 0.38305428624153137,
  'eval_accuracy': 0.8386491557223265,
  'eval_f1': 0.8349328214971209,
  'eval_precision': 0.8546168958742633,
  'eval_recall': 0.8161350844277674,
  'eval_runtime': 3.0434,
  'eval_samples_per_second': 350.261,
  'eval_steps_per_second': 11.172,
  'epoch': 3.0},
 {'total': 67694596,
  'trainable': 739586,
  'trainable_percent': 1.0925332946813067})

Osservazioni LoRA dalla run:

- test accuracy: `0.8386`;
- test F1: `0.8349`;
- parametri addestrabili: `739586`, pari a circa `1.09%` del modello;
- best validation accuracy osservata nei checkpoint: `0.8433` all'epoca 2.

La cella di setup puo' mostrare il normale report di caricamento del modello: i pesi `UNEXPECTED` appartengono alla testa di language modeling del checkpoint originale, mentre i pesi `MISSING` sono la nuova testa di classificazione inizializzata da addestrare. L'helper `suppress_subprocess_reader_unicode_errors()` mantiene pulito l'output da eventuali warning Unicode non legati al training.

## Strategia B - Partial Freezing

Come confronto semplice, congeliamo embeddings e i primi 4 layer di DistilBERT. In questo modo aggiorniamo solo gli ultimi layer del backbone e la testa di classificazione.

Questa strategia non richiede PEFT, ma addestra molti piu' parametri di LoRA. E' utile come baseline efficiente per capire quanto si perde o guadagna rispetto a un metodo specializzato.

In [6]:
def freeze_model_init():
    return partial_freezing_sequence_classifier_init(frozen_layers=4)


freeze_args = build_training_arguments(
    output_dir=output_dir("distilbert_partial_freeze"),
    learning_rate=2e-5,
    train_batch_size=batch_size,
    eval_batch_size=batch_size,
    num_train_epochs=num_epochs,
    weight_decay=0.01,
    warmup_steps=warmup_steps,
)

freeze_trainer = build_trainer(
    tokenizer=tokenizer,
    training_args=freeze_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    model_init=freeze_model_init,
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 6620.74it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
freeze_train_output = freeze_trainer.train()
disable_notebook_progress_callback(freeze_trainer)
freeze_test = freeze_trainer.evaluate(tokenized["test"])
freeze_params = count_parameters(freeze_trainer.model)
freeze_test, freeze_params

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 10629.79it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.96it/s]
There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.Layer

({'eval_loss': 0.39663007855415344,
  'eval_accuracy': 0.8377110694183865,
  'eval_f1': 0.8347659980897804,
  'eval_precision': 0.8501945525291829,
  'eval_recall': 0.8198874296435272,
  'eval_runtime': 2.8575,
  'eval_samples_per_second': 373.05,
  'eval_steps_per_second': 11.898,
  'epoch': 3.0},
 {'total': 66955010,
  'trainable': 14767874,
  'trainable_percent': 22.056413702275606})

Osservazioni partial freezing dalla run:

- test accuracy: `0.8377`;
- test F1: `0.8348`;
- parametri addestrabili: `14767874`, pari a circa `22.06%` del modello;
- best validation accuracy osservata nei checkpoint: `0.8358` all'epoca 2.

Partial freezing e' quindi efficiente rispetto al fine-tuning completo, ma molto meno parameter-efficient di LoRA.

In [8]:
comparison = pd.DataFrame(
    [
        {
            "method": "Full fine-tuning (Exercise 2)",
            "test_accuracy": 0.844278,
            "test_f1": 0.842803,
            "trainable_percent": 100.0,
        },
        {
            "method": "LoRA",
            "test_accuracy": lora_test.get("eval_accuracy"),
            "test_f1": lora_test.get("eval_f1"),
            "trainable_percent": lora_params["trainable_percent"],
        },
        {
            "method": "Partial freezing",
            "test_accuracy": freeze_test.get("eval_accuracy"),
            "test_f1": freeze_test.get("eval_f1"),
            "trainable_percent": freeze_params["trainable_percent"],
        },
    ]
)
comparison

,method,test_accuracy,test_f1,trainable_percent
0,Full fine-tuning (Exercise 2),0.844278,0.842803,100.000000
1,LoRA,0.838649,0.834933,1.092533
2,Partial freezing,0.837711,0.834766,22.056414


## Conclusioni Exercise 3.1

Tutti i punti richiesti sono stati svolti.

- Abbiamo trovato e testato un metodo efficiente basato su PEFT: **LoRA**.
- Abbiamo aggiunto un secondo confronto efficiente senza cambiare pipeline: **partial freezing**.
- Entrambi riusano dataset, tokenizer, data collator, metriche Scikit-learn e Trainer dei notebook precedenti.
- Il confronto include sia performance sia percentuale di parametri addestrabili.

Risultati osservati:

- full fine-tuning notebook 2: test accuracy `0.8443`, F1 `0.8428`, circa `100%` parametri addestrabili;
- LoRA: test accuracy `0.8386`, F1 `0.8349`, circa `1.09%` parametri addestrabili;
- partial freezing: test accuracy `0.8377`, F1 `0.8348`, circa `22.06%` parametri addestrabili.

Interpretazione: LoRA e partial freezing perdono circa `0.006` punti di accuracy rispetto al fine-tuning completo, ma LoRA addestra solo circa `1.09%` dei parametri. Per questo LoRA e' la scelta migliore dell'esercizio: mantiene performance molto vicine al full fine-tuning con un costo di aggiornamento molto piu' basso.

Best practice adottate:

- stesso protocollo dati e metriche dei notebook 1 e 2;
- valutazione su test solo dopo il training;
- confronto con baseline full fine-tuning;
- reporting esplicito di parametri addestrabili;
- uso di `warmup_steps` invece di `warmup_ratio` deprecato;
- rimozione/soppressione di callback e traceback notebook non rilevanti per mantenere output puliti.

## Referenced functions and source files

| Function/class | Defined in | Purpose |
| --- | --- | --- |
| `load_rotten_tomatoes` | `src/dla_lab2/sentiment.py` | Caricamento dataset sentiment. |
| `extract_cls_features_with_pipeline` | `src/dla_lab2/sentiment.py` | Feature extraction da DistilBERT. |
| `build_trainer` | `src/dla_lab2/sentiment.py` | Configurazione Hugging Face Trainer. |
| `CLIPAdapter` | `src/dla_lab2/clip_utils.py` | Adattatore leggero per feature CLIP. |
